<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/eda-%26-validation/notebooks/03_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validation:

- Design a validation strategy based on common sense and expected correlation with Public leaderboard scores.
- Explain your motivation for the chosen strategy.
- Discuss possible reasons for low or high correlation with leaderboard results.
- Perform adversarial validation if necessary (make sure to do it with the same model as the main task)

#### Data Leakage

- All texts with the same doc_id must be in the same fold.

- Duplicate texts must not appear in different folds.

#### Training Data

- UK and RU datasets have different sizes. We plan to use mostly UK data with a smaller RU subset.

- Some RU annotations are incorrect. We need to select cleaner examples.

#### Fold Distribution

- Keep the number of locations per text similarly distributed across folds.

- Make sure multiword locations are represented in each fold.

- Keep text length distributions similar across folds.

- Test texts are much longer than train texts, which may affect leaderboard scores.

#### Reproducibility

- Use a fixed seed and save the folds.

- All models should use the same validation split.

## Environment Setup

In [3]:
import os
import ast
import re
import hashlib
import shutil
import random
from pathlib import Path
from collections import Counter, defaultdict

import sys
import base64
import subprocess

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold

In [2]:
ram = psutil.virtual_memory()
disk = shutil.disk_usage("/content")

print(f"CPU cores: {os.cpu_count()}")
print(f"Total RAM: {ram.total / 1024**3:.1f} GB")
print(f"Available RAM: {ram.available / 1024**3:.1f} GB")
print(f"Free disk: {disk.free / 1024**3:.1f} GB")
print(f"Disk total: {disk.total / 1024**3:.1f} GB")

CPU cores: 8
Total RAM: 51.0 GB
Available RAM: 49.3 GB
Free disk: 225.7 GB
Disk total: 225.8 GB


In [4]:
if "google.colab" in sys.modules:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    root = "/content"
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    root = "/kaggle/working"

repo = "UA-Locations-NER-Deep-Learning"
branch = "eda-&-validation"
repo_dir = f"{root}/{repo}"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", "-b", branch, f"https://github.com/dkalchenko01/{repo}.git", repo_dir],
        env=env, check=True
    )

print("Repository:", repo_dir)

Repository: /content/UA-Locations-NER-Deep-Learning


In [5]:
sys.path.insert(0, repo_dir)
from src.data_io import get_data_dir

ENVIRONMENT = "colab" if root == "/content" else "kaggle"

data_dir = Path(get_data_dir(ENVIRONMENT))
print("Data directory:", data_dir)

for path in sorted(data_dir.iterdir()):
    print(path.name, "DIR" if path.is_dir() else "FILE")

100%|██████████| 811M/811M [00:05<00:00, 146MB/s]

Extracting files...


Data directory: /root/.cache/kagglehub/competitions/kse-ua-location-extraction-2026
README.md FILE
labeling_sample.csv FILE
ru_geo_dataset.csv FILE
test.csv FILE
uk_geo_dataset.csv FILE
uk_geo_dataset_processed_v1.parquet FILE


In [6]:
uk_df = pd.read_csv(data_dir / "uk_geo_dataset.csv")
ru_df = pd.read_csv(data_dir / "ru_geo_dataset.csv")

for name, df in [("UK", uk_df), ("RU", ru_df)]:
    print(f"{name}: {len(df):,} rows")
    print(f"DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

print(f"\nAvailable RAM: {psutil.virtual_memory().available / 1024**3:.2f} GB")

UK: 1,010,000 rows
DataFrame memory: 0.41 GB
RU: 8,028,840 rows
DataFrame memory: 3.63 GB

Available RAM: 45.65 GB


## Duplicates analysis

Symbol to symbol:

In [7]:
for name, df in [("UK", uk_df), ("RU", ru_df)]:
    duplicates = df["text"].duplicated().sum()
    print(f"{name}: {duplicates:,} duplicates ({duplicates / len(df):.2%})")

UK: 2,454 duplicates (0.24%)
RU: 523,501 duplicates (6.52%)


In [8]:
shared = uk_df["text"].isin(ru_df["text"]).sum()
print(f"UK rows also found in RU: {shared:,} ({shared / len(uk_df):.2%})")

UK rows also found in RU: 0 (0.00%)


Checking duplicate annotations


In [9]:
for name, df in [("UK", uk_df), ("RU", ru_df)]:
    repeated = df[df["text"].duplicated(keep=False)]
    conflicts = repeated.groupby("text")["loc_markers"].nunique().gt(1).sum()
    print(f"{name}: {conflicts:,} texts with different annotations")

UK: 0 texts with different annotations
RU: 4,786 texts with different annotations


In [10]:
conflicts = ru_df.groupby("text")["loc_markers"].nunique()
conflict_texts = conflicts[conflicts > 1].index

removed = ru_df["text"].isin(conflict_texts).sum()
print(f"Conflicting texts: {len(conflict_texts):,}")
print(f"Rows to remove: {removed:,} ({removed / len(ru_df):.2%})")

Conflicting texts: 4,786
Rows to remove: 22,562 (0.28%)


Decision: Exclude ru texts with conflicting annotations after verifying annotation equivalence

Similar texts, not identical copies:

In [11]:
#ignoring register and extra whitescapes
for name, df in [("UK", uk_df), ("RU", ru_df)]:
    normalized = df["text"].str.lower().str.split().str.join(" ")
    exact = df["text"].duplicated().sum()
    total = normalized.duplicated().sum()
    print(f"{name}: {total - exact:,} additional normalized duplicates")

UK: 65 additional normalized duplicates
RU: 2,134 additional normalized duplicates


Decision: leave one of the initial lines together with its own `loc_markers`

Near duplicates:

In [13]:
%pip install -q datasketch

In [23]:
from datasketch import MinHash, MinHashLSH

def shingles(text, k=3):
    words = re.sub(r"\W+", " ", text.lower()).split()
    return {" ".join(words[i:i+k]) for i in range(max(1, len(words)-k+1))}

def minhash(text, k=3, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in shingles(text, k):
        m.update(s.encode("utf8"))
    return m

In [24]:
uk_texts = uk_df["text"].dropna().drop_duplicates()

lsh_uk = MinHashLSH(threshold=0.7, num_perm=128)
pairs_uk = set()

for i, t in uk_texts.items():
    h = minhash(t)

    for j in lsh_uk.query(h):
        pairs_uk.add((int(j), i))

    lsh_uk.insert(str(i), h)

print(f"Texts checked: {len(uk_texts):,}")
print(f"Candidate pairs: {len(pairs_uk):,}")
print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.2f} GB")

Texts checked: 1,007,546
Candidate pairs: 6,356
Available RAM: 32.66 GB


In [25]:
similarities_uk = []

for i, j in pairs_uk:
    A = shingles(uk_texts.loc[i], k=3)
    B = shingles(uk_texts.loc[j], k=3)
    score = len(A & B) / len(A | B)
    similarities_uk.append((i, j, score))

scores_uk = pd.DataFrame(similarities_uk, columns=["id1", "id2", "jaccard"])

print(scores_uk["jaccard"].describe())
print(f"Pairs with Jaccard >= 0.7: {(scores_uk['jaccard'] >= 0.7).sum():,}")

count    6356.000000
mean        0.782601
std         0.162568
min         0.142857
25%         0.666667
50%         0.785714
75%         0.916667
max         1.000000
Name: jaccard, dtype: float64
Pairs with Jaccard >= 0.7: 4,382


Real examples:

In [26]:
for score in [0.7, 0.8, 0.9]:
    examples = scores_uk[scores_uk["jaccard"] >= score].sort_values("jaccard").head(2)

    print(f"\nJaccard >= {score}")
    for row in examples.itertuples():
        print(f"Similarity: {row.jaccard:.3f}")
        print("A:", uk_texts.loc[row.id1])
        print("B:", uk_texts.loc[row.id2])
        print()


Jaccard >= 0.7
Similarity: 0.700
A: Усик провів 17 поєдинків у професіоналах і здобув 17 перемог (13 нокаутом), 16 з них - у першій важкій вазі.
B: Усик провів 17 поєдинків у професіоналах і здобув 17 перемог (13 нокаутом), 16 із них - у першій важкій вазі.

Similarity: 0.700
A: Мальме - Динамо Київ у прямому ефірі дивіться на нашому сайті.
B: Шахтар Донецьк - Динамо Київ у прямому ефірі дивіться на нашому сайті.


Jaccard >= 0.8
Similarity: 0.800
A: Про це повідомляє офіційний сайт президента України.
B: Про це повідомляє офіційний сайт президента…

Similarity: 0.800
A: Центр контролю та профілактики захворювань у США, медичний інститут Національної академії наук Великобританії та британська Національна служба охорони здоров’я провели дослідження та спростували будь-який зв’язок між вакциною і виникненням аутизму.
B: Центр контролю та профілактики захворювань у США, медичний інститут Національної академії наук Великої Британії та британська Національна служба охорони здоров’я провели

In [27]:
confirmed_uk = scores_uk[scores_uk["jaccard"] >= 0.7]

affected_uk = set(confirmed_uk["id1"]) | set(confirmed_uk["id2"])

print(f"Confirmed pairs: {len(confirmed_uk):,}")
print(f"Unique texts involved: {len(affected_uk):,}")
print(f"Share of UK unique texts: {len(affected_uk) / len(uk_texts):.2%}")

Confirmed pairs: 4,382
Unique texts involved: 8,198
Share of UK unique texts: 0.81%


**Conclusions:**

UK: MinHash + LSH (3-word shingles, Jaccard ≥ 0.7) found 4,382 similar pairs covering 8,198 unique texts (0.81%).

- Near-duplicates are kept as is (original `text` and `loc_markers`).
- Related texts get a shared `near_dup_group_id`, used together with `doc_id` to keep them in the same fold.
- Group IDs live in the validation manifest only, not as model features.

RU: search deferred, since MinHash + LSH is approximate and not guaranteed to find all pairs.

Documents in ru dataset:

In [29]:
ru_doc_sizes = ru_df.groupby("doc_id").size().sort_values(ascending=False)

print("Top 10 largest documents:")
print(ru_doc_sizes.head(10))

print("\nDocuments with > 100 texts:", (ru_doc_sizes > 100).sum())

Top 10 largest documents:
doc_id
654996    686
630253    288
587165    136
421160    125
652192    123
236654    111
690521    110
444396    108
700709    104
442576    103
dtype: int64

Documents with > 100 texts: 10


## Preprocessing before train/val split example

In [30]:
def drop_conflicting_annotations(df):
    conflicts = df.groupby("text")["loc_markers"].transform("nunique") > 1
    return df.loc[~conflicts].copy()

In [32]:
def drop_duplicate_texts(df):
    normalized = df["text"].str.lower().str.split().str.join(" ")
    return df.loc[~normalized.duplicated()].copy()

In [34]:
def preprocess_data(df):
    df = drop_conflicting_annotations(df)
    df = drop_duplicate_texts(df)
    return df

In [35]:
uk_clean = preprocess_data(uk_df)
ru_clean = preprocess_data(ru_df)

In [36]:
for name, original, clean in [("UK", uk_df, uk_clean), ("RU", ru_df, ru_clean)]:
    print(f"\n{name}")
    print(f"Original: {len(original):,}")
    print(f"Clean: {len(clean):,}")
    print(f"Removed: {len(original) - len(clean):,}")
    print(f"Indexes preserved: {clean.index.isin(original.index).all()}")


UK
Original: 1,010,000
Clean: 1,007,481
Removed: 2,519
Indexes preserved: True

RU
Original: 8,028,840
Clean: 7,498,457
Removed: 530,383
Indexes preserved: True


Groups for UK near-duplicates

In [37]:
confirmed_uk = scores_uk[scores_uk["jaccard"] >= 0.7]

remaining_pairs_uk = confirmed_uk[
    confirmed_uk["id1"].isin(uk_clean.index) &
    confirmed_uk["id2"].isin(uk_clean.index)
].copy()

print(f"Confirmed pairs before cleaning: {len(confirmed_uk):,}")
print(f"Pairs after cleaning: {len(remaining_pairs_uk):,}")
print(f"Removed pairs: {len(confirmed_uk) - len(remaining_pairs_uk):,}")

Confirmed pairs before cleaning: 4,382
Pairs after cleaning: 4,313
Removed pairs: 69


In [38]:
parent = {i: i for i in uk_clean.index}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for a, b in remaining_pairs_uk[["id1", "id2"]].itertuples(index=False, name=None):
    union(a, b)

uk_clean["near_dup_group_id"] = [find(i) for i in uk_clean.index]

In [39]:
group_sizes = uk_clean.groupby("near_dup_group_id").size()

print(f"Total groups: {len(group_sizes):,}")
print(f"Groups with >1 text: {(group_sizes > 1).sum():,}")
print(f"Texts in multi-text groups: {group_sizes[group_sizes > 1].sum():,}")
print(f"Largest group: {group_sizes.max():,}")

Total groups: 1,003,326
Groups with >1 text: 3,915
Texts in multi-text groups: 8,070
Largest group: 18


## Checking `labeling_sample.csv`

In [40]:
labeling_df = pd.read_csv(get_data_dir("colab") / "labeling_sample.csv")

def parse_markers(value):
    if isinstance(value, str):
        return ast.literal_eval(value)
    return [] if pd.isna(value) else value

markers = labeling_df["loc_markers"].apply(parse_markers)
loc_counts = markers.apply(len)
lengths = labeling_df["text"].str.len()

print("Shape:", labeling_df.shape)
print("Columns:", labeling_df.columns.tolist())
print("Missing texts:", labeling_df["text"].isna().sum())
print("Duplicate texts:", labeling_df["text"].duplicated().sum())
print("Texts with locations:", (loc_counts > 0).sum())
print("Texts without locations:", (loc_counts == 0).sum())
print("Total location spans:", loc_counts.sum())
print("\nText length statistics:")
print(lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1))

Shape: (26, 3)
Columns: ['text', 'locations', 'loc_markers']
Missing texts: 0
Duplicate texts: 0
Texts with locations: 16
Texts without locations: 10
Total location spans: 32

Text length statistics:
count      26.0
mean      294.4
std       289.2
min         6.0
25%        76.2
50%       217.5
75%       433.5
95%       814.0
max      1118.0
Name: text, dtype: float64


`labeling_sample.csv` would be used only for errors diagnostics, but not as stand-alone validation set.

Datasets intersections:

In [41]:
test_df = pd.read_csv(get_data_dir("colab") / "test.csv")

datasets = {
    "UK": uk_clean,
    "RU": ru_clean,
    "labeling_sample": labeling_df,
    "test": test_df
}

texts = {
    name: set(df["text"].dropna())
    for name, df in datasets.items()
}

for a, b in [
    ("UK", "RU"),
    ("UK", "labeling_sample"),
    ("RU", "labeling_sample"),
    ("UK", "test"),
    ("RU", "test"),
    ("labeling_sample", "test")
]:
    print(f"{a} ↔ {b}: {len(texts[a] & texts[b]):,}")

UK ↔ RU: 0
UK ↔ labeling_sample: 0
RU ↔ labeling_sample: 0
UK ↔ test: 0
RU ↔ test: 0
labeling_sample ↔ test: 0


## Validation Strategy

- We use a fixed 80/20 train-validation split for Ukrainian data with `GroupShuffleSplit` (seed 17), grouping near-duplicate texts by `near_dup_group_id` to prevent leakage.
- We will check the balance of location-positive texts, entity counts, multiword locations and text lengths across training and validation. If substantial imbalances are found, we will consider group-aware stratification.
- russian data is split separately by `doc_id`. Different ru training subsets (100k, 300k, 500k) are compared using the same validation set.
- Because of the domain shift found in EDA, we also build a test-like version of the validation set (concatenated sentences with emojis, URLs and hashtags, same labels). The gap between the two scores shows how sensitive a model is to the shift.
- All models use the same fixed validation split for hyperparameter tuning. A 3-fold CV is planned only for the final configuration to assess stability.
- Exact duplicate checks found no overlaps between UK, RU, `labeling_sample` and test datasets.

### Correlation with Leaderboard Results

- We use the competition's normalized set-based micro F1 as the main evaluation metric.
- Validation focuses on Ukrainian data because Kaggle test is approximately 90% Ukrainian.
- EDA showed a domain shift: training data contains short sentences, while test contains longer Telegram-style posts with emojis, URLs and line breaks.
- The 26 annotated `labeling_sample` texts are reserved for qualitative error analysis, not model selection.
- We will compare local validation scores with Kaggle leaderboard results and investigate substantial differences, potentially using adversarial validation.
- Possible reasons for low correlation with the leaderboard: the test set has only 477 texts (noisy LB), the domain shift and different annotation conventions (addresses, streets) in test and train.

### UK validation split

In [42]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=17)
train_idx, val_idx = next(splitter.split(uk_clean, groups=uk_clean["near_dup_group_id"]))

uk_train = uk_clean.iloc[train_idx]
uk_val = uk_clean.iloc[val_idx]

print("Train:", len(uk_train), f"({len(uk_train) / len(uk_clean):.1%})")
print("Validation:", len(uk_val), f"({len(uk_val) / len(uk_clean):.1%})")
print("Shared groups:", len(set(uk_train["near_dup_group_id"]) & set(uk_val["near_dup_group_id"])))

Train: 805983 (80.0%)
Validation: 201498 (20.0%)
Shared groups: 0


Location balance:

In [43]:
def count_locations(df):
    return df["loc_markers"].apply(lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else len(x))

train_counts = count_locations(uk_train)
val_counts = count_locations(uk_val)

balance = pd.DataFrame({
    "Train": [(train_counts > 0).mean(), train_counts.mean()],
    "Validation": [(val_counts > 0).mean(), val_counts.mean()]
}, index=["Location-positive rate", "Mean locations per text"])

display(balance.round(4))

,Train,Validation
Location-positive rate,0.2311,0.2319
Mean locations per text,0.3416,0.3434


Text length balance:

In [44]:
length_balance = pd.DataFrame({
    "Train": uk_train["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95]),
    "Validation": uk_val["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95])
})

display(length_balance.round(1))

,Train,Validation
count,805983.0,201498.0
mean,102.1,102.3
std,46.6,46.6
min,39.0,40.0
50%,91.0,91.0
90%,170.0,171.0
95%,197.0,198.0
max,255.0,255.0


Location count and multiword location balance:

In [45]:
def location_stats(df):
    markers = df["loc_markers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    counts = markers.apply(len)
    multiword = [
        any(len(text[start:end].split()) > 1 for start, end in spans)
        for text, spans in zip(df["text"], markers)
    ]
    return counts.clip(upper=4).value_counts(normalize=True).sort_index(), sum(multiword) / len(df)

train_dist, train_multi = location_stats(uk_train)
val_dist, val_multi = location_stats(uk_val)

distribution = pd.DataFrame({
    "Train": train_dist,
    "Validation": val_dist
}).fillna(0).rename(index={4: "4+"})

display((distribution * 100).round(2))
display(pd.DataFrame({
    "Train": [train_multi * 100],
    "Validation": [val_multi * 100]
}, index=["Texts with multiword locations (%)"]).round(2))

,Train,Validation
loc_markers,,
0,76.89,76.81
1,15.94,16.03
2,4.87,4.79
3,1.46,1.49
4+,0.85,0.88


,Train,Validation
Texts with multiword locations (%),4.66,4.75


The checks confirmed:

- No shared groups between train and validation.

- Nearly identical location-positive rates (23.11% vs. 23.19%).

- Similar distributions of entity counts, multiword locations (4.66% vs. 4.75%) and text lengths.

Therefore, StratifiedGroupKFold would add unnecessary complexity without a clear benefit for our fixed holdout strategy.

### UK Validation manifest

Index check

In [46]:
print("Original index:", uk_df.index[:5].tolist())
print("Clean index:", uk_clean.index[:5].tolist())

print("Original index unique:", uk_df.index.is_unique)
print("Clean index unique:", uk_clean.index.is_unique)
print("Train index unique:", uk_train.index.is_unique)
print("Validation index unique:", uk_val.index.is_unique)

print("Clean indices preserved:", uk_clean.index.isin(uk_df.index).all())
print("Train/validation overlap:", len(uk_train.index.intersection(uk_val.index)))
print("All clean rows covered:", len(uk_train) + len(uk_val) == len(uk_clean))

Original index: [0, 1, 2, 3, 4]
Clean index: [0, 1, 2, 3, 4]
Original index unique: True
Clean index unique: True
Train index unique: True
Validation index unique: True
Clean indices preserved: True
Train/validation overlap: 0
All clean rows covered: True


In [47]:
uk_manifest = pd.DataFrame({
    "row_id": uk_clean.index,
    "source": "uk",
    "group_id": uk_clean["near_dup_group_id"].to_numpy(),
    "split": "train",
    "seed": 17
})

uk_manifest.loc[uk_manifest["row_id"].isin(uk_val.index), "split"] = "val"

display(uk_manifest.head())
print("\nSplit counts:")
print(uk_manifest["split"].value_counts())
print("Unique row_id:", uk_manifest["row_id"].is_unique)

,row_id,source,group_id,split,seed
0,0,uk,0,train,17
1,1,uk,1,train,17
2,2,uk,2,train,17
3,3,uk,3,train,17
4,4,uk,4,train,17



Split counts:
split
train    805983
val      201498
Name: count, dtype: int64
Unique row_id: True


### ru validation split

We use about 50,000 ru texts for validation because we only need to check how well the model works on russian data.

In [48]:
ru_val_target = 50_000
ru_val_fraction = ru_val_target / len(ru_clean)

ru_splitter = GroupShuffleSplit(n_splits=1, test_size=ru_val_fraction, random_state=17)
ru_train_idx, ru_val_idx = next(ru_splitter.split(ru_clean, groups=ru_clean["doc_id"]))

ru_train = ru_clean.iloc[ru_train_idx]
ru_val = ru_clean.iloc[ru_val_idx]

print("Train:", len(ru_train), f"({len(ru_train) / len(ru_clean):.2%})")
print("Validation:", len(ru_val), f"({len(ru_val) / len(ru_clean):.2%})")
print("Shared groups:", len(set(ru_train["doc_id"]) & set(ru_val["doc_id"])))

Train: 7449088 (99.34%)
Validation: 49369 (0.66%)
Shared groups: 0


Same balance checks as for UK:

In [49]:
ru_train_counts = count_locations(ru_train)
ru_val_counts = count_locations(ru_val)

ru_balance = pd.DataFrame({
    "Train": [(ru_train_counts > 0).mean(), ru_train_counts.mean()],
    "Validation": [(ru_val_counts > 0).mean(), ru_val_counts.mean()]
}, index=["Location-positive rate", "Mean locations per text"])

ru_train_dist, ru_train_multi = location_stats(ru_train)
ru_val_dist, ru_val_multi = location_stats(ru_val)

ru_distribution = pd.DataFrame({
    "Train": ru_train_dist,
    "Validation": ru_val_dist
}).fillna(0).rename(index={4: "4+"})

ru_lengths = pd.DataFrame({
    "Train": ru_train["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95]),
    "Validation": ru_val["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95])
})

print("1. Location balance")
display(ru_balance.round(4))

print("2. Location count distribution (%)")
display((ru_distribution * 100).round(2))

print("3. Multiword locations (%)")
display(pd.DataFrame({
    "Train": [ru_train_multi * 100],
    "Validation": [ru_val_multi * 100]
}, index=["Texts with multiword locations"]).round(2))

print("4. Text lengths")
display(ru_lengths.round(1))

1. Location balance


,Train,Validation
Location-positive rate,0.3086,0.3138
Mean locations per text,0.4618,0.4733


2. Location count distribution (%)


,Train,Validation
loc_markers,,
0,69.14,68.62
1,20.98,21.19
2,6.76,6.91
3,2.00,2.11
4+,1.12,1.17


3. Multiword locations (%)


,Train,Validation
Texts with multiword locations,4.97,4.97


4. Text lengths


,Train,Validation
count,7449088.0,49369.0
mean,121.3,122.4
std,58.7,59.0
min,1.0,1.0
50%,112.0,113.0
90%,197.0,198.0
95%,227.0,228.0
max,1316.0,865.0


All ru validation balance checks passed

### ru validation manifest

For the initial mixed-language model, we use 100,000 ru training texts to keep training manageable. We may increase this number later if time allows.

In [50]:
print("Original index unique:", ru_df.index.is_unique)
print("Clean index unique:", ru_clean.index.is_unique)
print("Train index unique:", ru_train.index.is_unique)
print("Validation index unique:", ru_val.index.is_unique)

print("Clean indices preserved:", ru_clean.index.isin(ru_df.index).all())
print("Train/validation overlap:", len(ru_train.index.intersection(ru_val.index)))
print("All clean rows covered:", len(ru_train) + len(ru_val) == len(ru_clean))

Original index unique: True
Clean index unique: True
Train index unique: True
Validation index unique: True
Clean indices preserved: True
Train/validation overlap: 0
All clean rows covered: True


In [51]:
ru_manifest = pd.DataFrame({
    "row_id": ru_clean.index,
    "source": "ru",
    "group_id": ru_clean["doc_id"].to_numpy(),
    "split": "train",
    "seed": 17
})

ru_manifest.loc[ru_manifest["row_id"].isin(ru_val.index), "split"] = "val"

display(ru_manifest.head())
print("\nSplit counts:")
print(ru_manifest["split"].value_counts())
print("Unique row_id:", ru_manifest["row_id"].is_unique)

,row_id,source,group_id,split,seed
0,0,ru,0,train,17
1,1,ru,0,train,17
2,2,ru,0,train,17
3,3,ru,0,train,17
4,4,ru,0,train,17



Split counts:
split
train    7449088
val        49369
Name: count, dtype: int64
Unique row_id: True


In [52]:
RU_TRAIN_SIZE = 100_000
SEED = 17

ru_train_sample = ru_train.sample(n=RU_TRAIN_SIZE, random_state=SEED)

ru_manifest = ru_manifest[
    (ru_manifest["split"] == "val") |
    (ru_manifest["row_id"].isin(ru_train_sample.index))
].copy()

print(ru_manifest["split"].value_counts())
print("Unique row_id:", ru_manifest["row_id"].is_unique)
print("Shared groups:", len(
    set(ru_train_sample["doc_id"]) & set(ru_val["doc_id"])
))

split
train    100000
val       49369
Name: count, dtype: int64
Unique row_id: True
Shared groups: 0


### Combined manifest

In [53]:
validation_manifest = pd.concat([uk_manifest, ru_manifest], ignore_index=True)

print("Manifest shape:", validation_manifest.shape)
print("Unique source + row_id:", ~validation_manifest.duplicated(["source", "row_id"]).any())
print("\nSplit counts:")
display(pd.crosstab(validation_manifest["source"], validation_manifest["split"]))

Manifest shape: (1156850, 5)
Unique source + row_id: True

Split counts:


split,train,val
source,,
ru,100000,49369
uk,805983,201498


### Test-like validation set

In [58]:
TESTLIKE_SEED = 17
TESTLIKE_SIZE = 5_000

uk_testlike_original = uk_val.sample(
    n=TESTLIKE_SIZE,
    random_state=TESTLIKE_SEED
).copy()

uk_testlike_original["row_id"] = uk_testlike_original.index

print("Selected texts:", len(uk_testlike_original))
print("Unique row_id:", uk_testlike_original["row_id"].is_unique)
print("All from UK validation:", uk_testlike_original.index.isin(uk_val.index).all())
print("Texts with locations:", (count_locations(uk_testlike_original) > 0).sum())

display(uk_testlike_original[["row_id", "text", "loc_markers"]].head())

Selected texts: 5000
Unique row_id: True
All from UK validation: True
Texts with locations: 1173


,row_id,text,loc_markers
991408,991408,В мережі з’явилося відео наїзду російського Ка...,[]
640103,640103,Згодом очікують на повернення інших тварин.,[]
69719,69719,"Здавалося б, сучасна епоха мала демонструвати ...",[]
640013,640013,"Віце-прем'єр-міністр, глава ради партії НСНУ Р...",[]
839874,839874,Подібні хімічні реакції стартують ще вдень: пі...,"[(62, 68)]"


In [59]:
rng = random.Random(17)
separators = ["\n", "\n\n", " 📍\n"]
noise = [" 🔔 #новини", " 📢 #Україна", " 🔗 https://example.com"]

def parse_spans(value):
    return ast.literal_eval(value) if isinstance(value, str) else value

def make_testlike(group):
    parts, spans, row_ids = [], [], []
    offset = 0

    for _, row in group.iterrows():
        sep = rng.choice(separators) if parts else ""
        parts.append(sep + row["text"])
        offset += len(sep)

        spans.extend((start + offset, end + offset)
                     for start, end in parse_spans(row["loc_markers"]))

        offset += len(row["text"])
        row_ids.append(int(row["row_id"]))

    suffix = rng.choice(noise)
    return {
        "text": "".join(parts) + suffix,
        "loc_markers": spans,
        "source_row_ids": row_ids
    }

testlike_rows = [
    make_testlike(uk_testlike_original.iloc[i:i + 5])
    for i in range(0, len(uk_testlike_original), 5)
]

uk_testlike = pd.DataFrame(testlike_rows)

print("Test-like texts:", len(uk_testlike))
print("Original sentences used:", sum(map(len, uk_testlike["source_row_ids"])))
display(uk_testlike.head(3))

Test-like texts: 1000
Original sentences used: 5000


,text,loc_markers,source_row_ids
0,В мережі з’явилося відео наїзду російського Ка...,"[(460, 466)]","[991408, 640103, 69719, 640013, 839874]"
1,"Так, для 50% охочих емігрувати мотивом є забез...","[(152, 159)]","[398431, 629670, 760612, 1001404, 858506]"
2,На Луганщині п'ятьох бійців з пораненнями допр...,"[(3, 12), (68, 74), (367, 375), (378, 388)]","[337417, 728570, 905883, 874037, 64910]"


In [60]:
original_spans = []
testlike_spans = []

for _, row in uk_testlike_original.iterrows():
    spans = parse_spans(row["loc_markers"])
    original_spans.extend(row["text"][start:end] for start, end in spans)

for _, row in uk_testlike.iterrows():
    testlike_spans.extend(
        row["text"][start:end] for start, end in row["loc_markers"]
    )

print("Original locations:", len(original_spans))
print("Test-like locations:", len(testlike_spans))
print("All locations match:", original_spans == testlike_spans)
print("Invalid spans:", sum(
    not (0 <= start < end <= len(row["text"]))
    for _, row in uk_testlike.iterrows()
    for start, end in row["loc_markers"]
))

Original locations: 1699
Test-like locations: 1699
All locations match: True
Invalid spans: 0
